# Day 5 — the flip manoeuvre

Four experiments against the 6-DoF flip optimiser.

| | |
|---|---|
| **A** | Fuel versus entry pitch, and where the ceiling is |
| **B** | When in the burn does the rotation actually happen |
| **C** | The flip tax, against Day 4's non-rotating solution |
| **D** | How small a gimbal can still fly the flip |

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from src.landing_flip import solve_flip_landing, feasible_entry_state
from src.dynamics_6dof import Vehicle6DoF

veh = Vehicle6DoF()
N, T_BURN = 80, 15.0
print(veh.summary())

## Experiment A — fuel versus entry pitch

The guide expects fuel to climb with entry tilt. It does — but the more
interesting result is that the sweep stops. Past a certain angle there is no
trajectory at all.

In [ ]:
angles = [0, 10, 20, 30, 40, 50, 60, 70]
fuel_a, ok_a = [], []
for a in angles:
    r = solve_flip_landing(N=N, t_burn=T_BURN, theta0_deg=float(a), verbose=False)
    good = r['status'].startswith('optimal')
    ok_a.append(good)
    fuel_a.append(r['fuel'] if good else np.nan)
    print(f"  theta0={a:3d} deg  "
          + (f"fuel {r['fuel']:>8,.0f} kg" if good else f"[{r['status']}]"))

plt.figure(figsize=(7,4))
plt.plot(angles, fuel_a, 'o-', lw=2)
plt.xlabel('Entry pitch from vertical [deg]'); plt.ylabel('Fuel [kg]')
plt.title('Fuel vs entry pitch (gap = infeasible)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**The ceiling.** The engine is lit for the whole trajectory — minimum throttle
is 40% and there is no coast — so while the vehicle is tilted, `sin(theta)` of
a very large thrust pushes it sideways whether it wants that or not. The pitch
rate is capped, so the flip takes at least `theta0 / omega_max` seconds, and the
lateral excursion built up in that window has to fit inside the glideslope
corridor *and* be nulled by touchdown.

A real Starship flips **before** the landing burn, unpowered, on aerodynamic
surfaces. That is exactly the freedom this model does not have, and it is why
a full 90° belly-flop is unreachable here.

In [ ]:
# Confirm the mechanism: raise the pitch rate limit and the ceiling moves.
for w_deg in (28.6, 40.0, 51.0, 70.0):
    v = Vehicle6DoF(omega_max=np.radians(w_deg))
    hi = None
    for a in (20, 30, 40, 50, 60, 70, 80):
        r = solve_flip_landing(vehicle=v, N=N, t_burn=T_BURN,
                               theta0_deg=float(a), verbose=False)
        if r['status'].startswith('optimal'):
            hi = a
        elif hi is not None:
            break
    print(f"  omega_max {w_deg:5.1f} deg/s  ->  highest feasible entry pitch {hi} deg")

## Experiment B — when does the rotation happen?

In [ ]:
r = solve_flip_landing(N=N, t_burn=T_BURN, theta0_deg=40.0, verbose=False)
t, th = r['t'], np.degrees(r['theta'])

# Time to cover 90% of the total attitude change.
travelled = np.abs(th - th[0])
i90 = int(np.argmax(travelled >= 0.9*abs(th[0] - th[-1])))
print(f"90% of the rotation is done by t = {t[i90]:.2f} s "
      f"({100*t[i90]/r['t_burn']:.0f}% of the burn)")
print(f"peak pitch rate {np.degrees(np.max(np.abs(r['omega']))):.1f} deg/s "
      f"(limit {np.degrees(veh.omega_max):.1f})")

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(t, th, lw=2); ax[0].axhline(0, color='k', lw=0.5)
ax[0].axvline(t[i90], color='r', ls='--', alpha=0.6, label='90% rotated')
ax[0].set_xlabel('Time [s]'); ax[0].set_ylabel('Pitch [deg]')
ax[0].set_title('Attitude'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(t, np.degrees(r['omega']), lw=2, color='tab:orange')
for s in (1,-1):
    ax[1].axhline(s*np.degrees(veh.omega_max), color='r', ls=':', alpha=0.6)
ax[1].set_xlabel('Time [s]'); ax[1].set_ylabel('Pitch rate [deg/s]')
ax[1].set_title('Rate (dotted = limit)'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

The rotation is **rate-limited, not torque-limited**: the pitch rate pins to its
bound almost immediately and stays there until the attitude is nearly nulled.
Peak torque is well under the maximum. Giving this vehicle a stronger gimbal
would not flip it faster — only raising `omega_max` would.

Note also the attitude does not settle monotonically. It overshoots past
vertical and comes back, because the optimiser is using attitude as its only
*steering* control: tilting the other way is the sole means of cancelling the
sideways velocity the flip itself produced.

## Experiment C — the flip tax

Same burn and entry geometry, with and without a rotation to perform.

In [ ]:
z0, vz0 = feasible_entry_state(veh, T_BURN, 40.0,
                               t_flip=1.4*np.radians(40.0)/veh.omega_max)
r_flip = solve_flip_landing(N=N, t_burn=T_BURN, theta0_deg=40.0, verbose=False)
r_up = solve_flip_landing(N=N, t_burn=T_BURN, theta0_deg=0.0,
                          z0=z0, vz0=vz0, verbose=False)
tax = r_flip['fuel'] - r_up['fuel']
print(f"  with flip    : {r_flip['fuel']:>8,.0f} kg")
print(f"  upright entry: {r_up['fuel']:>8,.0f} kg")
print(f"  flip tax     : {tax:>8,.0f} kg ({100*tax/r_up['fuel']:+.1f}%)")

## Experiment D — how small a gimbal still works?

In [ ]:
for d in (3, 5, 8, 10, 15, 20, 30):
    v = Vehicle6DoF(delta_max_deg=float(d))
    r = solve_flip_landing(vehicle=v, N=N, t_burn=T_BURN, theta0_deg=40.0,
                           verbose=False)
    if r['status'].startswith('optimal'):
        print(f"  gimbal {d:2d} deg -> fuel {r['fuel']:>8,.0f} kg, "
              f"peak gimbal used {np.degrees(np.max(np.abs(r['delta']))):5.1f} deg, "
              f"peak rate {np.degrees(np.max(np.abs(r['omega']))):5.1f} deg/s")
    else:
        print(f"  gimbal {d:2d} deg -> [{r['status']}]")

Read the *peak gimbal used* column against the limit. Where the vehicle stops
using its full deflection, the gimbal has ceased to be the binding constraint
and the pitch-rate cap has taken over — which is the same conclusion Experiment
B reached from the rate trace.